# 09. LangGraph 의심 거래 조사

Supabase 시연 거래의 과거 근거를 분석하고 Groq의 openai/gpt-oss-120b로 설명을 생성한다. 계좌 식별자와 원시 이력은 외부 전송에서 제외한다. API 키가 없거나 호출·검증에 실패하면 기존 규칙 기반 보고서를 반환한다. 인용 검증은 설명의 의미적 정확성을 보장하지 않으므로 담당자가 검토한다.


In [1]:
import json
import sys
from pathlib import Path
from uuid import uuid4

from fastapi.testclient import TestClient
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from fds_agent.agent.investigation import build_graph  # noqa: E402
from fds_agent.api.main import app  # noqa: E402

client = TestClient(app)
identity = "b677a82f-3ce6-5fdf-8309-6f06b1befee3"
response = client.post(f"/v1/transactions/{identity}/investigate")
assert response.status_code == 200, response.status_code
report = response.json()
assert report["status"] == "review_required"
assert all(row["step"] < report["cutoff_step"] for row in report["history"])
evidence = {identity} | {row["id"] for row in report["history"]}
assert all(set(f["evidence_ids"]) <= evidence for f in report["findings"])
assert client.post(f"/v1/transactions/{uuid4()}/investigate").status_code == 404
display(
    Markdown("### 조사 결과\n" + "\n".join("- " + f["description"] for f in report["findings"]))
)
display(report)
out = ROOT / "artifacts/reports" / ("investigation-" + uuid4().hex[:8])
out.mkdir(parents=True)
(out / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
(out / "graph.mmd").write_text(build_graph().get_graph().draw_mermaid())
print("Report:", out)

### 조사 결과
- 저장 모델의 위험 점수가 임계값 이상임을 확인한다.
- 거래 금액이 송금자의 거래 전 잔액과 일치함을 확인한다.

{'transaction_id': 'b677a82f-3ce6-5fdf-8309-6f06b1befee3',
 'status': 'review_required',
 'risk_score': 0.999999999999995,
 'threshold': 0.00635036124517738,
 'model_run_id': '20261119T073653Z-1d0e70df',
 'cutoff_step': 12,
 'findings': [{'code': 'model_alert',
   'evidence_ids': ['b677a82f-3ce6-5fdf-8309-6f06b1befee3'],
   'description': '저장 모델의 위험 점수가 임계값 이상임을 확인한다.'},
  {'code': 'full_balance_transfer',
   'evidence_ids': ['b677a82f-3ce6-5fdf-8309-6f06b1befee3'],
   'description': '거래 금액이 송금자의 거래 전 잔액과 일치함을 확인한다.'}],
 'history': [{'id': '3f81027a-7fa2-544e-af2e-c49a07dd7aed',
   'step': 11,
   'name_orig': 'demo_v1_alice',
   'name_dest': 'demo_v1_bob',
   'type': 'TRANSFER',
   'amount': 1000.0},
  {'id': '52e7231e-807e-54b1-bb5d-963e02dc6ad3',
   'step': 10,
   'name_orig': 'demo_v1_alice',
   'name_dest': 'demo_v1_shop',
   'type': 'PAYMENT',
   'amount': 200.0},
  {'id': 'c95aed1b-0b51-5e69-a356-48cc2bee04f9',
   'step': 10,
   'name_orig': 'demo_v1_alice',
   'name_dest': 'demo

Report: /Users/yueun/Desktop/FDS-AI-Agent/artifacts/reports/investigation-6863f167


In [2]:
from IPython.display import HTML
from html import escape

# 조사 흐름과 실제 설명 생성 여부를 함께 표시한다.
flow = " → ".join(report["trace"])
display(HTML("<div>" + escape(flow) + "</div>"))
print("설명 생성 상태:", report["explanation_status"])
print("생성 방식:", report["generation_mode"])
if report["explanation"]:
    for claim in report["explanation"]["claims"]:
        print(claim["explanation"])
        print("근거 ID:", ", ".join(claim["evidence_ids"]))
else:
    print("LLM 설명을 생성하지 않아 기존 조사 보고서를 사용한다.")


설명 생성 상태: generated
생성 방식: llm_assisted
저장 모델의 위험 점수가 임계값 이상임을 확인한다.
근거 ID: b677a82f-3ce6-5fdf-8309-6f06b1befee3
거래 금액이 송금자의 거래 전 잔액과 일치함을 확인한다.
근거 ID: b677a82f-3ce6-5fdf-8309-6f06b1befee3


## 저장 결과와 추가 사례 검증

저장된 보고서를 재조회하고 정상 합성 거래의 실제 설명을 검증한다. 계좌 정보가 부족하면 LLM을 호출하지 않는지 검증한다. 추가 사례는 메모리에서 구성하며 실제 거래나 정확도 평가 자료로 해석하지 않는다.

In [3]:
saved = client.get(f"/v1/investigations/{report['report_id']}")
assert saved.status_code == 200
assert saved.json() == report
assert client.get(f"/v1/investigations/{uuid4()}").status_code == 404
print("저장 보고서 재조회 일치를 확인한다.")

# 정상 합성 거래에서는 이전 수취 사실만 설명하도록 검증한다.
normal = dict(id="synthetic-normal", step=12, name_orig="synthetic-a",
              name_dest="synthetic-b", features={"amount": 10, "oldbalanceOrg": 100},
              risk_score=0.001, threshold=0.01, is_fraud=False, model_run_id="synthetic")
past = dict(id="synthetic-past", step=11, name_orig="synthetic-c",
            name_dest="synthetic-a", amount=50)
normal_report = build_graph(lambda _: (normal, [past])).invoke(
    {"transaction_id": normal["id"]}
)["report"]
assert normal_report["status"] == "no_model_alert"
assert normal_report["explanation_status"] == "generated"
assert normal_report["risk_score"] == normal["risk_score"]
assert normal_report["explanation"]["claims"][0]["evidence_ids"] == [past["id"]]
display(normal_report["explanation"])

# 계좌 정보가 없으면 외부 모델 호출을 생략하도록 검증한다.
def unexpected_call(_):
    raise AssertionError("근거 부족 사례에서는 LLM 호출을 생략한다.")

missing = {**normal, "name_orig": None}
missing_report = build_graph(lambda _: (missing, []), generator=unexpected_call).invoke(
    {"transaction_id": missing["id"]}
)["report"]
assert missing_report["explanation_status"] == "insufficient_evidence"
print("계좌 정보 부족 시 LLM 호출 생략을 확인한다.")


저장 보고서 재조회 일치를 확인한다.


{'claims': [{'finding_code': 'prior_incoming',
   'explanation': '송금 계좌에 이전에 수취된 거래가 1건 존재함을 확인한다.',
   'evidence_ids': ['synthetic-past']}]}

계좌 정보 부족 시 LLM 호출 생략을 확인한다.
